# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.6: Simulación de Proteínas y Biomoléculas

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/06_simulacion_proteinas.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Ejecutar un protocolo completo de simulación DM de una proteína globular
- Seleccionar y aplicar el campo de fuerza adecuado para biomoléculas
- Realizar la secuencia de minimización → NVT → NPT → producción
- Monitorear la estabilidad de la simulación en tiempo real
- Realizar simulaciones de membranas lipídicas y complejos proteína-ligando
- Usar GROMACS y/o OpenMM para simulaciones de producción

---

## 1. Instalación de Dependencias

In [ ]:
!pip install numpy matplotlib mdanalysis requests biopython
# GROMACS: conda install -c conda-forge gromacs
# OpenMM:  conda install -c conda-forge openmm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Bibliotecas importadas correctamente")

## 2. Campos de Fuerza para Biomoléculas

El campo de fuerza es la piedra angular de una simulación de DM. Define todas las interacciones entre átomos.

### Componentes de un Campo de Fuerza

$$U_{total} = U_{enlace} + U_{ángulo} + U_{torsión} + U_{VdW} + U_{electrostática}$$

$$U_{enlace} = \sum_{enlaces} \frac{k_b}{2}(r - r_0)^2$$

$$U_{ángulo} = \sum_{ángulos} \frac{k_\theta}{2}(\theta - \theta_0)^2$$

$$U_{torsión} = \sum_{dihedros} \frac{V_n}{2}[1 + \cos(n\phi - \delta)]$$

$$U_{VdW} = \sum_{i<j} 4\varepsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right]$$

$$U_{elec} = \sum_{i<j} \frac{q_i q_j}{4\pi\varepsilon_0 r_{ij}}$$

In [ ]:
# Comparar campos de fuerza principales
import pandas as pd

campos_fuerza = {
    'Campo de Fuerza': [
        'AMBER14SB', 'AMBER99SB-ILDN', 'CHARMM36m',
        'CHARMM22*', 'GROMOS54a7', 'OPLS-AA'
    ],
    'Familia': ['AMBER', 'AMBER', 'CHARMM', 'CHARMM', 'GROMOS', 'OPLS'],
    'Agua recomendada': [
        'TIP3P-FB / OPC', 'TIP3P', 'TIP3P (CHARMM)',
        'TIP3P', 'SPC/E', 'TIP4P'
    ],
    'Mejor para': [
        'Proteínas, péptidos', 'Proteínas', 'Proteínas, lípidos, glúcidos',
        'Proteínas IDPs', 'Proteínas, lípidos', 'Proteínas, moléculas orgánicas'
    ],
    'Software': ['AMBER, OpenMM, GROMACS', 'GROMACS, AMBER', 'NAMD, GROMACS', 'NAMD', 'GROMACS', 'GROMACS, AMBER']
}
df_ff = pd.DataFrame(campos_fuerza)
print("Principales Campos de Fuerza para Biomoléculas")
print("=" * 80)
print(df_ff.to_string(index=False))

## 3. Protocolo Completo con GROMACS: Lisozima en Agua

In [ ]:
# Script completo de simulación GROMACS
script_gromacs = """
#!/bin/bash
# ============================================================
# SIMULACIÓN COMPLETA GROMACS - Lisozima en agua
# ============================================================

set -e  # Parar si hay error

PDB="1AKI_limpio"
FF="amber99sb-ildn"
WATER="spce"

echo "[1/7] Generando topología con $FF"
gmx pdb2gmx -f ${PDB}.pdb -o processed.gro \\
    -water $WATER -ff $FF -ignh

echo "[2/7] Definiendo caja dodecaédrica (borde 1.0 nm)"
gmx editconf -f processed.gro -o box.gro \\
    -c -d 1.0 -bt dodecahedron

echo "[3/7] Solvatación"
gmx solvate -cp box.gro -cs spc216.gro \\
    -o solv.gro -p topol.top

echo "[4/7] Añadiendo iones (neutralizar + 0.15 M NaCl)"
gmx grompp -f mdp/ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo "SOL" | gmx genion -s ions.tpr -o solv_ions.gro \\
    -p topol.top -pname NA -nname CL -neutral -conc 0.15

echo "[5/7] Minimización de energía"
gmx grompp -f mdp/em.mdp    -c solv_ions.gro -p topol.top -o em.tpr
gmx mdrun  -v -deffnm em

echo "[6/7] Equilibración NVT (100 ps)"
gmx grompp -f mdp/nvt.mdp   -c em.gro  -r em.gro  -p topol.top -o nvt.tpr
gmx mdrun  -v -deffnm nvt

echo "[7/7] Equilibración NPT (100 ps)"
gmx grompp -f mdp/npt.mdp   -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
gmx mdrun  -v -deffnm npt

echo "[8/8] Producción MD (10 ns)"
gmx grompp -f mdp/md.mdp    -c npt.gro -t npt.cpt -p topol.top -o md.tpr
gmx mdrun  -v -deffnm md -ntmpi 1 -ntomp 4  # 1 GPU, 4 CPU hilos

echo "✓ Simulación completada!"
"""

Path('scripts').mkdir(exist_ok=True)
with open('scripts/run_md.sh', 'w') as f:
    f.write(script_gromacs)
print("Script guardado en: scripts/run_md.sh")
print()
print(script_gromacs)

## 4. Protocolo con OpenMM

OpenMM permite escribir simulaciones completas en Python, facilitando la automatización y el análisis.

In [ ]:
# Protocolo completo con OpenMM
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
    from sys import stdout

    # Cargar sistema ya preparado (con agua e iones)
    pdb = app.PDBFile('estructuras_dm/1AKI_openmm.pdb')
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

    system = forcefield.createSystem(
        pdb.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometer,
        constraints=app.HBonds
    )

    # --- PASO 1: Minimización ---
    minimizer = mm.VerletIntegrator(0.001*unit.picoseconds)
    sim_min = app.Simulation(pdb.topology, system, minimizer)
    sim_min.context.setPositions(pdb.positions)
    print("Energía antes de minimizar:",
          sim_min.context.getState(getEnergy=True).getPotentialEnergy())
    sim_min.minimizeEnergy(maxIterations=1000)
    print("Energía después de minimizar:",
          sim_min.context.getState(getEnergy=True).getPotentialEnergy())
    posiciones_min = sim_min.context.getState(getPositions=True).getPositions()

    # --- PASO 2: NVT con termostato de Langevin ---
    integrador_nvt = mm.LangevinMiddleIntegrator(
        300*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds
    )
    sim_nvt = app.Simulation(pdb.topology, system, integrador_nvt)
    sim_nvt.context.setPositions(posiciones_min)
    sim_nvt.context.setVelocitiesToTemperature(300*unit.kelvin)

    sim_nvt.reporters.append(app.StateDataReporter(
        stdout, 5000, step=True, temperature=True, potentialEnergy=True
    ))
    sim_nvt.step(50000)  # 100 ps NVT
    posiciones_nvt = sim_nvt.context.getState(getPositions=True).getPositions()

    # --- PASO 3: NPT con barostato Monte Carlo ---
    system.addForce(mm.MonteCarloBarostat(1*unit.bar, 300*unit.kelvin))
    integrador_npt = mm.LangevinMiddleIntegrator(
        300*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds
    )
    sim_npt = app.Simulation(pdb.topology, system, integrador_npt)
    sim_npt.context.setPositions(posiciones_nvt)
    sim_npt.context.setVelocitiesToTemperature(300*unit.kelvin)
    sim_npt.reporters.append(app.StateDataReporter(
        'md_log.txt', 5000,
        step=True, time=True, temperature=True,
        potentialEnergy=True, density=True
    ))
    sim_npt.reporters.append(app.DCDReporter('traj.dcd', 5000))
    sim_npt.step(500000)  # 1 ns producción
    print("✓ Simulación NPT completada")

except ImportError:
    print("OpenMM no disponible. El código anterior es un ejemplo de referencia.")
except Exception as e:
    print(f"Error: {e}")
    print("Asegúrate de haber completado la Actividad 5.5 primero.")

## 5. Simulación de un Complejo Proteína-Ligando

Para simular complejos proteína-ligando se necesita un proceso adicional: **parametrización del ligando**.

In [ ]:
# Flujo para complejo proteína-ligando
flujo_complejo = """
Flujo para complejo proteína-ligando:
======================================

1. Obtener estructura del complejo (PDB)
   - Separar proteína y ligando en archivos distintos

2. Preparar proteína
   - pdb2gmx con AMBER14SB

3. Parametrizar ligando (GAFF2)
   - Generar cargas RESP con Antechamber (AMBER):
     antechamber -i ligando.pdb -fi pdb -o ligando.mol2 -fo mol2 -c bcc -s 2
     parmchk2 -i ligando.mol2 -f mol2 -o ligando.frcmod
   - Convertir a GROMACS con acpype:
     acpype -i ligando.mol2 -c bcc -n 0

4. Combinar topologías
   - Incluir itp del ligando en topol.top
   - Mezclar .gro de proteína + ligando con editconf

5. Continuar con protocolo estándar
   - Añadir agua e iones
   - Minimización → NVT → NPT → Producción

Herramientas útiles:
- ACPYPE: https://github.com/alanwilter/acpype
- LigParGen: https://ligpargen.org/
- CHARMM-GUI Ligand Reader: https://www.charmm-gui.org
"""
print(flujo_complejo)

## 6. Consideraciones de Rendimiento Computacional

| Sistema | Átomos | Hardware | Velocidad típica |
|---------|--------|----------|------------------|
| Proteína pequeña (~150 res) | ~40,000 | 1 GPU RTX 3080 | ~500 ns/día |
| Proteína mediana (~300 res) | ~80,000 | 1 GPU RTX 3080 | ~250 ns/día |
| Proteína grande (~500 res) | ~150,000 | 1 GPU A100 | ~200 ns/día |
| Complejo proteína-ADN | ~200,000 | 4 GPU A100 | ~300 ns/día |
| Membrana lipídica | ~100,000 | 1 GPU RTX 3080 | ~400 ns/día |

### Estrategias para acelerar simulaciones
- **Hydrogen mass repartitioning (HMR):** redistribuir masa hacia hidrógenos → dt = 4 fs
- **LINCS constraints:** restringir todos los enlaces con H
- **PME cutoff:** usar 1.0-1.2 nm (balance entre precisión y velocidad)
- **GPU acceleration:** GROMACS y OpenMM soportan CUDA/OpenCL

In [ ]:
# Visualizar escalado de rendimiento
n_atomos = np.array([10000, 20000, 50000, 100000, 200000, 500000])
# Rendimiento aproximado (ns/día) en 1 GPU RTX 3080
rendimiento_gpu   = 5000 / (n_atomos / 1000)**1.1
rendimiento_8cpu  = 800  / (n_atomos / 1000)**1.2

plt.figure(figsize=(9, 5))
plt.loglog(n_atomos, rendimiento_gpu,  'b-o', linewidth=2, markersize=8, label='1 GPU (RTX 3080)')
plt.loglog(n_atomos, rendimiento_8cpu, 'g-s', linewidth=2, markersize=8, label='8 CPU cores')
plt.xlabel('Número de átomos', fontsize=12)
plt.ylabel('Rendimiento (ns/día)', fontsize=12)
plt.title('Rendimiento de DM vs Tamaño del Sistema\n(GROMACS, estimación)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rendimiento_dm.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Ejercicios

### Ejercicio 1 (Básico)
Analiza el archivo `scripts/run_md.sh` y explica cada paso del protocolo. ¿Para qué sirve la restricción de posiciones (`-DPOSRES`) durante la equilibración NVT?

### Ejercicio 2 (Intermedio)
Modifica el protocolo GROMACS para simular una proteína de membrana. ¿Qué parámetros del archivo `.mdp` deben cambiarse? Considera: tipo de caja, barostato semianisótropo, campo de fuerza de los lípidos.

### Ejercicio 3 (Avanzado)
Usando OpenMM, implementa un protocolo de **calentamiento gradual** que vaya de 0 K a 300 K en incrementos de 50 K, con 20 ps de equilibración a cada temperatura. Registra la temperatura y la energía potencial durante todo el proceso.

In [ ]:
# Ejercicio 3 - Calentamiento gradual (simulación del protocolo)
np.random.seed(42)
temperaturas_objetivo = np.arange(0, 350, 50)  # 0, 50, 100, ..., 300 K
pasos_por_etapa = 20  # representación simplificada

T_registro = []
E_registro = []
etapas = []

T_actual = 0
E_actual = -45000

for T_obj in temperaturas_objetivo:
    for p in range(pasos_por_etapa):
        T_actual = T_actual + (T_obj - T_actual) * 0.15 + np.random.normal(0, 2)
        E_actual = E_actual + np.random.normal(0, 50) + (T_obj - 50) * 0.5
        T_registro.append(T_actual)
        E_registro.append(E_actual)
    etapas.append(len(T_registro) - pasos_por_etapa)

t_ps = np.arange(len(T_registro)) * 1  # 1 ps por paso

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(t_ps, T_registro, 'b-', linewidth=1.2)
for i, (T_obj, inicio) in enumerate(zip(temperaturas_objetivo, etapas)):
    axes[0].axvline(inicio, color='gray', linestyle=':', alpha=0.5)
    axes[0].text(inicio + 2, T_obj + 5, f'{T_obj}K', fontsize=9, color='red')
axes[0].set_ylabel('Temperatura (K)', fontsize=12)
axes[0].set_title('Protocolo de Calentamiento Gradual', fontsize=13)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_ps, E_registro, 'g-', linewidth=1.2)
axes[1].set_xlabel('Tiempo (ps)', fontsize=12)
axes[1].set_ylabel('Energía Potencial (kJ/mol)', fontsize=12)
axes[1].set_title('Energía Potencial durante Calentamiento', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calentamiento_gradual.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Recursos Adicionales

- **Tutoriales GROMACS:**
  - [Lysozyme in Water](http://www.mdtutorials.com/gmx/lysozyme/) — Justin Lemkul
  - [Protein-Ligand Complex](http://www.mdtutorials.com/gmx/complex/)
  - [GROMACS Tutorials Oficiales](https://tutorials.gromacs.org/)

- **OpenMM:**
  - [OpenMM Cook Book](https://openmm.org/documentation/latest/cookbook/)
  - [OpenMM Tutorials](https://openmm.org/documentation/latest/userguide/application/01_getting_started_with_openmm.html)

- **Parametrización de ligandos:**
  - [ACPYPE](https://github.com/alanwilter/acpype)
  - [LigParGen](https://ligpargen.org/)
  - [CHARMM-GUI Ligand Reader](https://www.charmm-gui.org/?doc=input/ligandrm)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Ejecutar el protocolo completo de simulación DM con GROMACS (pdb2gmx → mdrun)
- ✅ Seleccionar y aplicar el campo de fuerza adecuado para una proteína globular
- ✅ Realizar la secuencia minimización → NVT → NPT → producción
- ✅ Implementar el mismo protocolo con la API Python de OpenMM
- ✅ Monitorear la estabilidad de la simulación durante la producción

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.6: Simulación de Proteínas y Biomoléculas**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.5-Preparación_de_Sistemas-blue.svg)](05_preparacion_sistemas.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.7_➡️-Análisis_de_Trayectorias-green.svg)](07_analisis_trayectorias.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>